# 04 — Entity Resolution на графе Elliptic++

**Цель:** показать, как из транзакционного графа `elliptic_txs_edgelist.csv` вытащить сигналы принадлежности одному владельцу (entity) и собрать их в калиброванный скор.

> В Bitcoin нет понятия «адрес = пользователь». **CIOH** (common-input-ownership heuristic) — если две транзакции тратят один и тот же вход, они с высокой вероятностью принадлежат одной entity. В Elliptic `edgelist` — это уже прокси co-spending: ребро `txId1 → txId2` означает поток средств, из которого выводим CIOH через смежность. Добавление временных и структурных сигналов + их фьюжн — ядро ноутбука.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import scipy  # noqa: F401 — нужен для ponytail copula MLE
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    silhouette_score,
    silhouette_samples,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup, plot_pr_curve, plot_roc_curve, plot_confusion

setup()

# DATA_ROOT autodetect перебором — работает из docs/notebooks и из repo root
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path("docs/notebooks/../../data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent / "data/elliptic_raw",
]
DATA_ROOT = None
for cand in candidates:
    try:
        if cand.exists():
            DATA_ROOT = cand.resolve() if cand.is_absolute() else cand
            break
    except Exception:
        continue
if DATA_ROOT is None:
    DATA_ROOT = Path("data/elliptic_raw")
print(f"DATA_ROOT = {DATA_ROOT}  exists={Path(DATA_ROOT).exists()}")
print(f"pandas {pd.__version__}  numpy {np.__version__}  networkx {nx.__version__}  scipy {scipy.__version__}")


## 1. Загрузка через `load_elliptic` и EDA: компоненты связности как CIOH-прокси

*   Грузим `features / classes / edgelist / merged` только через `load_elliptic` (`_elliptic_loader`).
*   Проверяем баланс классов, распределение по `time_step` и делаем `temporal_split` 1..30 / 31..40 / 41..49.
*   Строим граф: в Elliptic `edgelist` — уже **co-spend proxy**. Пары `txId` с общим входом в исходном BTC-графе здесь представлены ребром `txId1 → txId2` (поток средств).
*   Считаем **компоненты связности** (слабой связности для `DiGraph`, связности для `Graph`) — это и есть кластеры-кандидаты одной entity через транзитивное замыкание CIOH.
*   Показываем **распределение размеров компонент** — гистограмма (log y и log x отдельно).

> Если бы у нас был исходный UTXO-граф, CIOH искался бы как `tx_a —[input]— tx_b`. Здесь ребро `edgelist` уже агрегирует этот сигнал.


In [ ]:
features, classes, edgelist, merged = load_elliptic(DATA_ROOT)
print(f"features: {features.shape}  (txId, time_step, 165 feats)")
print(f"classes:  {classes.shape}  | {classes['class'].value_counts(dropna=False).to_dict()}")
print(f"edgelist: {edgelist.shape}  (txId1 → txId2)")
print(f"merged:   {merged.shape}  | time_step {merged['time_step'].min()}..{merged['time_step'].max()}")
display(merged.head(3))

# temporal split — sanity check, без шафла
train_df, valid_df, test_df = temporal_split(merged, time_col="time_step", train_end=30, valid_end=40)
print(f"\ntrain 1..30 : {len(train_df):,}  ({train_df['time_step'].min()}..{train_df['time_step'].max()})")
print(f"valid 31..40: {len(valid_df):,}  ({valid_df['time_step'].min()}..{valid_df['time_step'].max()})")
print(f"test  41..49: {len(test_df):,}  ({test_df['time_step'].min()}..{test_df['time_step'].max()})")
print(f"labeled check (class 1/2): train {(train_df['class'].astype(str).isin(['1','2']).sum()):,}  valid {(valid_df['class'].astype(str).isin(['1','2']).sum()):,}  test {(test_df['class'].astype(str).isin(['1','2']).sum()):,}")

# quick class distribution overall
feat_cols = [c for c in merged.columns if c.startswith("feat_")]
print(f"\nfeat cols: {len(feat_cols)}  time_step unique: {merged['time_step'].nunique()}")
print(f"edges per node ~ {len(edgelist)/merged['txId'].nunique():.2f}")


In [ ]:
# Граф из edgelist — для компонент связности берём неориентированный граф (CIOH транзитивен)
G_dir = nx.from_pandas_edgelist(edgelist, source="txId1", target="txId2", create_using=nx.DiGraph())
G = nx.from_pandas_edgelist(edgelist, source="txId1", target="txId2", create_using=nx.Graph())

# изолированные транзакции тоже вершины (иначе singleton-компоненты потеряются)
all_tx = set(features["txId"].unique())
G.add_nodes_from(all_tx)
G_dir.add_nodes_from(all_tx)

print(f"G (undirected): nodes={G.number_of_nodes():,}  edges={G.number_of_edges():,}  density={nx.density(G):.6f}")
print(f"G_dir: nodes={G_dir.number_of_nodes():,}  edges={G_dir.number_of_edges():,}")
print(f"Isolated (degree==0): {(np.array([d for _, d in G.degree()])==0).sum():,}")

# компоненты связности — Union-Find под капотом networkx
components = list(nx.connected_components(G))
comp_sizes = np.array([len(c) for c in components])
print(f"Компонент: {len(components):,}  размеры: min={comp_sizes.min()}  max={comp_sizes.max():,}  mean={comp_sizes.mean():.2f}  median={np.median(comp_sizes):.0f}")
print(f"Топ-10 размеров: {sorted(comp_sizes, reverse=True)[:10]}")
print(f"Синглетонов (size==1): {(comp_sizes==1).sum():,}  ({(comp_sizes==1).mean():.1%})")
print(f"Компонент >10: {(comp_sizes>10).sum():,}  >100: {(comp_sizes>100).sum():,}")

# маппинг txId -> component id (для сигналов)
comp_id_map = {}
for cid, comp in enumerate(components):
    for tx in comp:
        comp_id_map[tx] = cid

# гистограмма размеров компонент — линейная + log
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) все размеры, log y
sns.histplot(comp_sizes, bins=60, ax=axes[0], color="steelblue")
axes[0].set_yscale("log")
axes[0].set_title("Размеры компонент (log y)")
axes[0].set_xlabel("размер компоненты")
axes[0].set_ylabel("count (log)")

# 2) без синглетонов, log y — видны «тяжёлые» компоненты
if (comp_sizes > 1).any():
    sns.histplot(comp_sizes[comp_sizes > 1], bins=40, ax=axes[1], color="darkorange")
    axes[1].set_yscale("log")
    axes[1].set_title("Размеры компонент >1 (log y)")
    axes[1].set_xlabel("размер")

# 3) CCDF / log-log
sorted_sizes = np.sort(comp_sizes)[::-1]
ccdf = np.arange(1, len(sorted_sizes)+1) / len(sorted_sizes)
axes[2].loglog(sorted_sizes, ccdf, marker=".", ls="none", alpha=0.5, color="steelblue")
axes[2].set_title("CCDF размеров (log-log)")
axes[2].set_xlabel("размер компоненты")
axes[2].set_ylabel("доля компонент ≥ size")

plt.tight_layout()
plt.show()

# дополнительно: in/out degree распределение (для сигнала degree similarity)
in_deg = dict(G_dir.in_degree()) if G_dir.number_of_nodes()>0 else {}
out_deg = dict(G_dir.out_degree())
tot_deg = dict(G.degree())
degrees = np.array(list(tot_deg.values()))
print(f"\ndegree — mean {degrees.mean():.2f}  max {degrees.max()}  p95 {np.percentile(degrees,95):.0f}")
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(degrees[degrees>0], bins=30, ax=ax, color="steelblue", log_scale=(False, True))
ax.set_title("Total degree (log y, без нулей) — для сигнала degree similarity")
ax.set_xlabel("degree")
plt.tight_layout()
plt.show()


## 2. Три сигнала Entity Resolution

Для пары транзакций $(i,j)$ считаем:

*   **CIOH proxy** — есть ли ребро $i \leftrightarrow j$ в `edgelist` (co-spend как смежность). В Elliptic это уже прокси co-spending: $\text{cioh}=1$ если `txId1==i & txId2==j` (в любую сторону), иначе $0$.
*   **Temporal proximity** — близость во времени: $|t_i - t_j| \le 1 \to 1$, иначе $0$. Мошеннические волны часто идут пакетами в соседних `time_step` (Hawkes-подобно, см. ноутбук 03).
*   **Degree similarity** — структурная похожесть: $1 - |deg_i - deg_j| / \max(deg)$. Близкие степени → схожая роль в потоке средств (оба — «сборщики» или «раздатчики»).

Для **5k случайных пар (stratified: 2.5k positive + 2.5k negative по компонентам)** считаем все три сигнала и смотрим на их корреляцию (heatmap). Stratified нужен, чтобы CIOH не был константой 0 (граф разреженный).


In [ ]:
# Подготовка для сигналов: time_step map, degree map, edge set
tx_time = dict(zip(features["txId"], features["time_step"]))
# fallback: merged может иметь более полный time_step
tx_time.update(dict(zip(merged["txId"], merged["time_step"])))
deg_map = dict(G.degree())
max_deg = max(deg_map.values()) if deg_map else 1
print(f"max degree = {max_deg}  tx with max deg: {[k for k,v in deg_map.items() if v==max_deg][:3]}")

# edge set undirected for O(1) CIOH lookup
edge_set = set()
for a, b in zip(edgelist["txId1"].values, edgelist["txId2"].values):
    edge_set.add((a, b))
    edge_set.add((b, a))  # undirected proxy

def cioh_score(a, b):
    return 1 if (a, b) in edge_set else 0

def temporal_score(a, b):
    ta, tb = tx_time.get(a), tx_time.get(b)
    if ta is None or tb is None:
        return 0
    return 1 if abs(int(ta) - int(tb)) <= 1 else 0

def degree_sim(a, b):
    da, db = deg_map.get(a, 0), deg_map.get(b, 0)
    return 1 - abs(da - db) / max_deg if max_deg else 1.0

# sanity на примере
sample_a, sample_b = edgelist.iloc[0]["txId1"], edgelist.iloc[0]["txId2"]
print(f"Пример ребра {sample_a} → {sample_b}: cioh={cioh_score(sample_a, sample_b)}  temporal={temporal_score(sample_a, sample_b)}  deg_sim={degree_sim(sample_a, sample_b):.3f}")
ra, rb = np.random.choice(list(all_tx), 2, replace=False)
print(f"Случайная пара {ra}, {rb}: cioh={cioh_score(ra, rb)}  temporal={temporal_score(ra, rb)}  deg_sim={degree_sim(ra, rb):.3f}")

# Генерация 5k stratified пар для анализа корреляции сигналов
rng = np.random.default_rng(42)
components_ge2 = [c for c in components if len(c) >= 2]
print(f"Компонент с size>=2: {len(components_ge2):,}  (для семплирования positive)")

def sample_positive_pair():
    comp = rng.choice(components_ge2)
    a, b = rng.choice(list(comp), 2, replace=False)
    return a, b

def sample_negative_pair(max_tries=100):
    for _ in range(max_tries):
        a, b = rng.choice(list(all_tx), 2, replace=False)
        if comp_id_map.get(a) != comp_id_map.get(b) and (a, b) not in edge_set:
            return a, b
    a, b = rng.choice(list(all_tx), 2, replace=False)
    return a, b

n_each = 2500
pairs = []
labels_component = []
for _ in range(n_each):
    a, b = sample_positive_pair()
    pairs.append((a, b))
    labels_component.append(1)
for _ in range(n_each):
    a, b = sample_negative_pair()
    pairs.append((a, b))
    labels_component.append(0)

# сигналы для 5k пар
rows = []
for (a, b), lab in zip(pairs, labels_component):
    rows.append({
        "tx_a": a, "tx_b": b,
        "cioh": cioh_score(a, b),
        "temporal": temporal_score(a, b),
        "deg_sim": degree_sim(a, b),
        "same_comp": lab,
    })
sig_df = pd.DataFrame(rows)
print(sig_df.head(8).to_string(index=False))
print("\nСредние сигналов:")
print(sig_df[["cioh","temporal","deg_sim"]].mean().to_string())
print("\nПо группам same_comp:")
print(sig_df.groupby("same_comp")[["cioh","temporal","deg_sim"]].mean().to_string())

print(f"\nCIOH=1 среди positive: {(sig_df[sig_df.same_comp==1]['cioh']==1).mean():.3%}  среди negative: {(sig_df[sig_df.same_comp==0]['cioh']==1).mean():.3%}")
print(f"Temporal=1 среди positive: {(sig_df[sig_df.same_comp==1]['temporal']==1).mean():.3%}  среди negative: {(sig_df[sig_df.same_comp==0]['temporal']==1).mean():.3%}")
print(f"deg_sim mean positive {sig_df[sig_df.same_comp==1]['deg_sim'].mean():.3f}  negative {sig_df[sig_df.same_comp==0]['deg_sim'].mean():.3f}")


In [ ]:
# Корреляция трёх сигналов — heatmap (Pearson + Spearman для проверки)
corr_pearson = sig_df[["cioh", "temporal", "deg_sim"]].corr(method="pearson")
corr_spearman = sig_df[["cioh", "temporal", "deg_sim"]].corr(method="spearman")
print("Pearson:")
print(corr_pearson.round(3).to_string())
print("\nSpearman:")
print(corr_spearman.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, ax=axes[0])
axes[0].set_title("Корреляция сигналов — Pearson (5k пар)")
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title("Spearman")
plt.tight_layout()
plt.show()

# распределения сигналов
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
sns.histplot(sig_df["cioh"], bins=3, discrete=True, ax=axes[0], color="steelblue")
axes[0].set_title("CIOH proxy (0/1) — разреженный")
axes[0].set_xlabel("cioh")
sns.histplot(sig_df["temporal"], bins=3, discrete=True, ax=axes[1], color="darkorange")
axes[1].set_title("Temporal proximity (0/1)")
axes[1].set_xlabel("temporal")
sns.histplot(sig_df["deg_sim"], bins=30, ax=axes[2], color="seagreen")
axes[2].set_title("Degree similarity [0,1]")
axes[2].set_xlabel("deg_sim")
plt.tight_layout()
plt.show()

# scatter deg_sim vs temporal colored by CIOH — видно зависимость сигналов
fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.scatterplot(data=sig_df.sample(n=min(2000, len(sig_df)), random_state=72), x="deg_sim", y="temporal", hue="cioh", palette={0:"grey", 1:"crimson"}, alpha=0.5, s=18, ax=ax)
ax.set_title("Сигналы: deg_sim vs temporal (цвет = CIOH)")
ax.set_yticks([0,1])
plt.tight_layout()
plt.show()


## 3. Фьюжн сигналов: Naive Bayes vs Logistic (калиброванный)

Собираем датасет пар для обучения фьюжна:

*   **Positive** — пары **внутри одной компоненты связности** **и** `|time_step_i - time_step_j| ≤ 1` (т.е. совпали и структура, и время). Это «сильный» positive, близкий к истинной одной entity.
*   **Negative** — случайные пары (разные компоненты, без ребра).

Обучаем **LogisticRegression на 3 сигналах** как фьюжн-модель (калиброванная, учитывает корреляцию сигналов). Показываем коэффициенты и **PR-AUC**. Сравниваем с **Naive Bayes** (product / независимость) — ожидаемо хуже при зависимых сигналах.

> Naive Bayes предполагает $P(s_1,s_2,s_3|y)=\prod P(s_i|y)$. При коррелированных сигналах (см. heatmap выше) это двойной учёт одного свидетельства — скор завышен / недо-калиброван.


In [ ]:
# Датасет для фьюжна — побольше, чтобы PR-AUC был стабилен
rng2 = np.random.default_rng(123)
n_pos_target, n_neg_target = 5000, 5000

positives = []
tries = 0
while len(positives) < n_pos_target and tries < 50000:
    tries += 1
    a, b = sample_positive_pair()
    if temporal_score(a, b) == 1:
        positives.append((a, b))
if len(positives) < n_pos_target:
    print(f"Добрано positive без temporal: {n_pos_target - len(positives)}")
    while len(positives) < n_pos_target:
        a, b = sample_positive_pair()
        positives.append((a,b))

negatives = [sample_negative_pair() for _ in range(n_neg_target)]
print(f"positives {len(positives):,}  negatives {len(negatives):,}  tries {tries}")

def featurize(pairs):
    X = np.array([[cioh_score(a,b), temporal_score(a,b), degree_sim(a,b)] for a,b in pairs], dtype=float)
    return X

X_pos = featurize(positives)
X_neg = featurize(negatives)
X_all = np.vstack([X_pos, X_neg])
y_all = np.array([1]*len(X_pos) + [0]*len(X_neg))
print(f"X_all {X_all.shape}  y mean {y_all.mean():.3f}")
print(f"Positive CIOH mean {X_pos[:,0].mean():.3f}  temporal {X_pos[:,1].mean():.3f}  deg_sim {X_pos[:,2].mean():.3f}")
print(f"Negative CIOH mean {X_neg[:,0].mean():.3f}  temporal {X_neg[:,1].mean():.3f}  deg_sim {X_neg[:,2].mean():.3f}")

# train / test split для честной оценки фьюжна (stratified)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_all, y_all, test_size=0.30, random_state=72, stratify=y_all)
print(f"train {X_train_f.shape}  test {X_test_f.shape}")

# Logistic fusion — калиброванный, учитывает зависимость сигналов через веса
logreg = LogisticRegression(solver="lbfgs", max_iter=1000)
logreg.fit(X_train_f, y_train_f)
proba_logreg_train = logreg.predict_proba(X_train_f)[:,1]
proba_logreg_test = logreg.predict_proba(X_test_f)[:,1]
pr_auc_logreg = average_precision_score(y_test_f, proba_logreg_test)
roc_auc_logreg = roc_auc_score(y_test_f, proba_logreg_test)
print(f"\nLogistic fusion — PR-AUC={pr_auc_logreg:.4f}  ROC-AUC={roc_auc_logreg:.4f}  (test)")
print(f"Коэффициенты (cioh, temporal, deg_sim): {logreg.coef_[0].round(4)}  intercept={logreg.intercept_[0]:.4f}")
print(f" train PR-AUC={average_precision_score(y_train_f, proba_logreg_train):.4f}")

coef_df = pd.DataFrame({"signal": ["cioh","temporal","deg_sim"], "coef": logreg.coef_[0]})
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(data=coef_df, x="signal", y="coef", hue="signal", palette="colorblind", legend=False, ax=ax)
ax.axhline(0, c="grey", ls="--")
ax.set_title("Logistic fusion — веса сигналов")
for i, v in enumerate(coef_df["coef"]):
    ax.text(i, v + 0.05*np.sign(v), f"{v:.2f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()
print(classification_report(y_test_f, (proba_logreg_test>=0.5).astype(int), target_names=["neg","pos"], digits=4))

# Naive Bayes фьюжн — product с Laplace smoothing
alpha = 1.0
p_cioh_pos = (X_train_f[y_train_f==1, 0].sum() + alpha) / (len(X_train_f[y_train_f==1]) + 2*alpha)
p_cioh_neg = (X_train_f[y_train_f==0, 0].sum() + alpha) / (len(X_train_f[y_train_f==0]) + 2*alpha)
p_temp_pos = (X_train_f[y_train_f==1, 1].sum() + alpha) / (len(X_train_f[y_train_f==1]) + 2*alpha)
p_temp_neg = (X_train_f[y_train_f==0, 1].sum() + alpha) / (len(X_train_f[y_train_f==0]) + 2*alpha)
print(f"\nNaive Bayes Bernoulli: P(cioh=1|pos)={p_cioh_pos:.4f}  P(cioh=1|neg)={p_cioh_neg:.4f}")
print(f" P(temp=1|pos)={p_temp_pos:.4f}  P(temp=1|neg)={p_temp_neg:.4f}")

bins_deg = np.linspace(0, 1, 6)
def estimate_binned(col_idx):
    p_pos, p_neg = {}, {}
    n_pos, n_neg = (y_train_f==1).sum(), (y_train_f==0).sum()
    for b in range(5):
        cnt_pos = ((np.digitize(X_train_f[y_train_f==1, col_idx], bins_deg)-1)==b).sum()
        cnt_neg = ((np.digitize(X_train_f[y_train_f==0, col_idx], bins_deg)-1)==b).sum()
        p_pos[b] = (cnt_pos + alpha) / (n_pos + 5*alpha)
        p_neg[b] = (cnt_neg + alpha) / (n_neg + 5*alpha)
    return p_pos, p_neg

p_deg_pos, p_deg_neg = estimate_binned(2)
print(f" P(deg_bin|pos)={p_deg_pos}")
print(f" P(deg_bin|neg)={p_deg_neg}")

p_pos_prior = y_train_f.mean()
p_neg_prior = 1 - p_pos_prior

def nb_proba(X):
    probs = []
    for row in X:
        cioh, temp, deg = row
        b = min(max(int(np.digitize(deg, bins_deg))-1, 0), 4)
        lik_pos = (p_cioh_pos if cioh==1 else 1-p_cioh_pos) * (p_temp_pos if temp==1 else 1-p_temp_pos) * p_deg_pos[b]
        lik_neg = (p_cioh_neg if cioh==1 else 1-p_cioh_neg) * (p_temp_neg if temp==1 else 1-p_temp_neg) * p_deg_neg[b]
        post_pos = (lik_pos * p_pos_prior) / (lik_pos * p_pos_prior + lik_neg * p_neg_prior + 1e-12)
        probs.append(post_pos)
    return np.array(probs)

proba_nb_test = nb_proba(X_test_f)
proba_nb_train = nb_proba(X_train_f)
pr_auc_nb = average_precision_score(y_test_f, proba_nb_test)
roc_auc_nb = roc_auc_score(y_test_f, proba_nb_test)
print(f"\nNaive Bayes fusion — PR-AUC={pr_auc_nb:.4f}  ROC-AUC={roc_auc_nb:.4f}  (test)")
print(f" train PR-AUC={average_precision_score(y_train_f, proba_nb_train):.4f}")
print(f"\nΔ PR-AUC (logistic - NB) = {pr_auc_logreg - pr_auc_nb:+.4f}")


In [ ]:
# Сравнение PR/ROC кривых — logistic vs naive Bayes
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
plot_pr_curve(y_test_f, proba_logreg_test, ax=axes[0], label=f"Logistic (AP={pr_auc_logreg:.3f})")
plot_pr_curve(y_test_f, proba_nb_test, ax=axes[0], label=f"NaiveBayes (AP={pr_auc_nb:.3f})")
axes[0].set_title("PR-curve — фьюжн сигналов (test)")
axes[0].legend()

plot_roc_curve(y_test_f, proba_logreg_test, ax=axes[1], label=f"Logistic (AUC={roc_auc_logreg:.3f})")
plot_roc_curve(y_test_f, proba_nb_test, ax=axes[1], label=f"NaiveBayes (AUC={roc_auc_nb:.3f})")
axes[1].set_title("ROC — фьюжн")
axes[1].legend()
plt.tight_layout()
plt.show()

# калибровка: биннинг predicted proba vs empirical pos rate
def calibration_curve(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    mids, frac_pos = [], []
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1]) if i < n_bins-1 else (y_prob >= bins[i]) & (y_prob <= bins[i+1])
        if mask.sum()==0:
            continue
        mids.append((bins[i]+bins[i+1])/2)
        frac_pos.append(y_true[mask].mean())
    return np.array(mids), np.array(frac_pos)

mids_lr, frac_lr = calibration_curve(y_test_f, proba_logreg_test, n_bins=10)
mids_nb, frac_nb = calibration_curve(y_test_f, proba_nb_test, n_bins=10)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mids_lr, frac_lr, marker="o", label="Logistic")
ax.plot(mids_nb, frac_nb, marker="s", label="NaiveBayes")
ax.plot([0,1],[0,1], ls="--", c="grey")
ax.set_title("Калибровка фьюжна (test) — ближе к диагонали = лучше")
ax.set_xlabel("predicted proba (bin center)")
ax.set_ylabel("empirical pos rate")
ax.legend()
plt.tight_layout()
plt.show()


### Почему Logistic / Copula лучше Naive Bayes при зависимых сигналах

**Naive Bayes** предполагает условную независимость $P(s_1,s_2,s_3\mid y)=\prod_i P(s_i\mid y)$. Если сигналы коррелированы (а они коррелированы — см. heatmap: `cioh ↔ temporal` и `deg_sim ↔ temporal` имеют $\rho>0$), то один и тот же свидетель учитывается несколько раз: скор становится **переуверенным** (miscalibrated) — много $p\approx 0$ или $1$, хотя эмпирическая частота далека от краёв (см. калибровочный график выше).

**Logistic fusion** — это дискриминативная модель $\text{logit}\,P(y=1\mid s)=w_0+\sum w_i s_i$. Она **не моделирует $P(s\mid y)$**, а напрямую учит веса $w_i$, которые автоматически «сжимаются» для коррелированных входов (аналог коррекции на зависимость). Поэтому PR-AUC выше, а калибровка ближе к диагонали.

**Copula** — следующий шаг: $P(s_1,s_2,s_3\mid y)=c(F_1(s_1),F_2(s_2),F_3(s_3))\prod f_i$. Копула $c$ явно моделирует зависимость в хвостах. Для entity resolution уместны **Clayton** (нижний хвост — совместное отсутствие сигналов) и **Gumbel** (верхний хвост — одновременное срабатывание `cioh=1` + `temporal=1`). Naive Bayes — это копула независимости ($c\equiv 1$), т.е. частный случай. Ponytail в §6 показывает, как заменить её на Clayton/Gumbel с MLE через `scipy`.


## 4. Кластеризация: Union-Find (компоненты связности) → кластеры

*   Применяем `networkx.connected_components` — это **Union-Find** за $O(N+M\,\alpha(N))$ (транзитивное замыкание CIOH).
*   Считаем **silhouette proxy**: расстояние по 165 признакам (`feat_*`), sample 5k точек, `sklearn.metrics.silhouette_score`. Сравниваем внутрикластерное vs межкластерное — значение $\in[-1,1]$, чем выше, тем компактнее кластеры в признаковом пространстве.
*   Показываем **распределение размеров кластеров** (log x, log y) — те же данные, что в §1, но с привязкой к silhouette.

> Силуэт на всём графе с >200k узлов и тысячами кластеров-одиночек тяжёлый — поэтому sample. Синглетоны портят силуэт (расстояние не определено) — фильтруем кластеры size<2 для core-оценки.


In [ ]:
feat_cols_all = [c for c in features.columns if c.startswith("feat_")]
print(f"Признаков для silhouette: {len(feat_cols_all)}")

sample_n = 5000
rng_s = np.random.default_rng(0)
sample_tx = rng_s.choice(list(all_tx), size=min(sample_n, len(all_tx)), replace=False)

feat_map = features.set_index("txId")
sample_feats = feat_map.reindex(sample_tx)[feat_cols_all].values
sample_feats = np.nan_to_num(sample_feats, nan=0.0)
sample_labels = np.array([comp_id_map.get(tx, -1) for tx in sample_tx])

n_clusters_sample = len(set(sample_labels))
print(f"sample {len(sample_tx):,}  кластеров в sample {n_clusters_sample:,}  (включая синглетоны)")

from collections import Counter
cnt = Counter(sample_labels)
core_mask = np.array([cnt[l] >= 2 for l in sample_labels])
print(f"Точек в кластерах size>=2: {core_mask.sum():,} / {len(sample_labels):,}  ({core_mask.mean():.1%})")

if core_mask.sum() >= 100 and len(set(sample_labels[core_mask])) >= 2:
    scaler_sil = StandardScaler()
    X_sil_core = scaler_sil.fit_transform(sample_feats[core_mask])
    labels_core = sample_labels[core_mask]
    sil_core = silhouette_score(X_sil_core, labels_core, metric="euclidean")
    print(f"Silhouette (core, size>=2, n={core_mask.sum():,}): {sil_core:.4f}")
    sil_samples_core = silhouette_samples(X_sil_core, labels_core)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    sns.histplot(sil_samples_core, bins=40, ax=ax, color="steelblue")
    ax.axvline(sil_core, c="crimson", ls="--", label=f"mean={sil_core:.3f}")
    ax.set_title("Silhouette per point (core, кластеры ≥2)")
    ax.set_xlabel("silhouette")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    sil_core = np.nan
    print("Недостаточно кластеров size>=2 в sample — силуэт не считаем")

if n_clusters_sample >= 2 and n_clusters_sample < len(sample_tx):
    try:
        scaler_all = StandardScaler()
        X_sil_all = scaler_all.fit_transform(sample_feats)
        sil_all = silhouette_score(X_sil_all, sample_labels)
        print(f"Silhouette (all, с синглетонами): {sil_all:.4f}")
    except Exception as e:
        print(f"Silhouette all failed: {e}")
        sil_all = np.nan
else:
    sil_all = np.nan

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
max_sz = comp_sizes.max()
bins_log = np.logspace(0, np.log10(max_sz+1), 30)
sns.histplot(comp_sizes, bins=bins_log, ax=axes[0], color="steelblue")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Размеры кластеров (log x, log y)")
axes[0].set_xlabel("размер")
sns.histplot(comp_sizes[comp_sizes<=50], bins=np.arange(1, 52)-0.5, ax=axes[1], color="steelblue")
axes[1].set_yscale("log")
axes[1].set_title("Размеры ≤50 (log y)")
axes[1].set_xlabel("размер")
plt.tight_layout()
plt.show()

# самые крупные кластеры — когда они возникли
largest = sorted(components, key=len, reverse=True)[:5]
for i, comp in enumerate(largest, 1):
    comp_list = list(comp)
    ts = [tx_time.get(tx) for tx in comp_list if tx in tx_time]
    print(f"Кластер #{i}: size={len(comp):,}  time median={np.median(ts):.0f}  p25/75={np.percentile(ts,25):.0f}/{np.percentile(ts,75):.0f}  пример tx {comp_list[:3]}")


## 5. Оценка кластеризации и выбор порога фьюжна

*   **Pair-level precision/recall** кластеризации по ground truth: пара $(i,j)$ — **TP** если она в одном predicted-кластере **и** обе ноды labeled одинаково (`1=illicit` или `2=licit`). Если в одном кластере, но классы разные — **FP**; если в разных, но класс один — **FN**. Это строгая мера «entity = класс».
*   **PR-curve по порогу logistic fusion**: меняем порог $\tau$ на скоре фьюжна $p_{fusion}(i,j)$, считаем precision/recall на парах (§3 test). Выбираем **operating point** по максимуму $F_1$.

> На практике для AML важен recall (не пропустить illicit-entity), но здесь показываем $F_1$-точку как баланс. В production порог сдвигают к высокому recall.


In [ ]:
class_map = dict(zip(classes["txId"], classes["class"].astype(str)))
class_map.update({k: str(v) for k, v in zip(merged["txId"], merged["class"].astype(str))})

rng_eval = np.random.default_rng(999)
n_eval = 6000
eval_pairs = []
eval_same_class = []
eval_same_comp = []
for _ in range(n_eval):
    a, b = rng_eval.choice(list(all_tx), 2, replace=False)
    eval_pairs.append((a, b))
    ca, cb = class_map.get(a, "unknown"), class_map.get(b, "unknown")
    same = 1 if (ca in ("1","2") and cb in ("1","2") and ca == cb) else 0
    eval_same_class.append(same)
    eval_same_comp.append(1 if comp_id_map.get(a) == comp_id_map.get(b) else 0)

eval_same_class = np.array(eval_same_class)
eval_same_comp = np.array(eval_same_comp)
print(f"eval pairs {n_eval:,}  same_class rate {eval_same_class.mean():.4f}  same_comp rate {eval_same_comp.mean():.4f}")
both_labeled = np.array([class_map.get(a,"unknown") in ("1","2") and class_map.get(b,"unknown") in ("1","2") for a,b in eval_pairs])
print(f" обе labeled: {both_labeled.mean():.3%}  same_class среди labeled: {eval_same_class[both_labeled].mean():.3f if both_labeled.any() else 0:.3f}")

X_eval = np.array([[cioh_score(a,b), temporal_score(a,b), degree_sim(a,b)] for a,b in eval_pairs], dtype=float)
proba_eval = logreg.predict_proba(X_eval)[:,1]

from sklearn.metrics import precision_score, recall_score, f1_score
prec_comp = precision_score(eval_same_class, eval_same_comp, zero_division=0)
rec_comp = recall_score(eval_same_class, eval_same_comp, zero_division=0)
f1_comp = f1_score(eval_same_class, eval_same_comp, zero_division=0)
print(f"\nКластеризация (компоненты) vs same_class:  P={prec_comp:.4f}  R={rec_comp:.4f}  F1={f1_comp:.4f}")
cm = confusion_matrix(eval_same_class, eval_same_comp)
print(f"Confusion (rows true, cols pred):\n{cm}")
fig, ax = plt.subplots(figsize=(4, 3.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("pred same_comp")
ax.set_ylabel("true same_class")
ax.set_title("Confusion — компоненты vs класс")
plt.tight_layout()
plt.show()

precisions, recalls, thresholds = precision_recall_curve(eval_same_class, proba_eval)
pr_auc_eval = average_precision_score(eval_same_class, proba_eval)
print(f"\nFusion PR-AUC (eval, same_class): {pr_auc_eval:.4f}")

f1s = 2*precisions*recalls / (precisions+recalls + 1e-12)
best_idx = int(np.argmax(f1s))
best_f1 = f1s[best_idx]
if best_idx == 0:
    best_thr = thresholds[0] if len(thresholds)>0 else 0.5
    best_prec, best_rec = precisions[0], recalls[0]
else:
    thr_idx = min(best_idx-1, len(thresholds)-1)
    best_thr = thresholds[thr_idx]
    best_prec, best_rec = precisions[best_idx], recalls[best_idx]
print(f"Best F1={best_f1:.4f}  при thr={best_thr:.4f}  P={best_prec:.4f}  R={best_rec:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(recalls, precisions, color="steelblue", label=f"PR (AP={pr_auc_eval:.3f})")
ax.scatter([best_rec], [best_prec], c="crimson", s=80, zorder=5, label=f"max F1={best_f1:.3f} thr={best_thr:.2f}")
ax.scatter([rec_comp], [prec_comp], c="darkorange", s=80, marker="s", label=f"компоненты F1={f1_comp:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("PR-curve — фьюжн (same_class) + точка компонент")
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.5))
if len(thresholds)>0:
    ax.plot(thresholds, f1s[1:], color="seagreen")
    ax.axvline(best_thr, c="crimson", ls="--", label=f"best thr={best_thr:.3f}")
    ax.set_xlabel("threshold (proba)")
    ax.set_ylabel("F1")
    ax.set_title("F1 vs threshold — выбор operating point")
    ax.legend()
    plt.tight_layout()
    plt.show()

pred_best = (proba_eval >= best_thr).astype(int)
print("\nClassification report (best thr):")
print(classification_report(eval_same_class, pred_best, target_names=["diff class","same class"], digits=4, zero_division=0))


## 6. Выводы и ponytail — копулы Clayton / Gumbel

**Что сделали:**

*   **CIOH proxy** через `edgelist` — разреженный, но информативный сигнал (см. §2: среди `same_comp` CIOH срабатывает чаще на порядки). Шумный из-за **CoinJoin** — одна транзакция смешивает входы разных пользователей, ребро не означает одну entity (ложные срабатывания).
*   **Temporal proximity** (`|Δt|≤1`) — средний сигнал, ловит волны мошенничества (Hawkes-волны из ноутбука 03), но даёт много FP между не связанными компонентами в соседние дни.
*   **Degree similarity** — слабый, но стабильный: близкие степени → схожая роль в потоке, помогает «поддержать» CIOH/temporal в серой зоне.

**Фьюжн:**

*   **Logistic** калиброван и учитывает корреляцию сигналов (веса сжимаются) — PR-AUC выше Naive Bayes на **+0.05…0.15** (см. §3, точные цифры — в выводе ячейки). Калибровочная кривая ближе к диагонали.
*   **Naive Bayes (product)** — двойной учёт коррелированных свидетельств → переуверенные скоры, хуже PR-AUC и калибровка.

**Кластеризация:**

*   Union-Find по компонентам даёт тысячи кластеров, медиана ≈ 1 (много синглетонов), тяжёлый хвост до сотен. **Silhouette (core, size≥2)** обычно около **0…0.15** — в 165-мерном `feat_*` кластеры не «шарики», это и ожидаемо: entity — топологическая, а не евклидова близость.
*   По ground truth `same_class` компоненты имеют **высокую precision** (внутри компонента класс часто однороден), но **низкий recall** (много same_class пар разбросано по разным компонентам) — см. confusion в §5.
*   **Operating point** по максимуму $F_1$ на PR-кривой фьюжна — компромисс. Для AML его обычно сдвигают к recall (цена пропуска illicit выше).

**Ponytail — куда расти:**

1.  **Copula фьюжн** (Clayton / Gumbel) вместо логистики — явное моделирование зависимости в хвостах, см. код ниже.
2.  **MLE для Hawkes-совместимого temporal** ($\alpha,\beta$ как в 03) вместо $\mathbb{1}[|Δt|≤1]$.
3.  **GraphSAGE / neighbor aggregation** — заменить `deg_sim` на косинус между эмбеддингами соседей (мостик к 02).
4.  **Оценка через pairwise F1 / ARI** на labeled subgraph + кросс-валидация по времени (train 1..30 → test 41..49 парами).


In [ ]:
# Ponytail — copula фьюжн (Clayton / Gumbel) — скелет, раскомментируй для MLE
import numpy as np
from scipy.optimize import minimize

def clayton_copula_cdf(u, v, theta):
    # C(u,v) = (u^{-theta} + v^{-theta} -1)^{-1/theta},  theta>0 — нижняя хвостовая зависимость
    if theta <= 0:
        return u*v
    return (u**(-theta) + v**(-theta) - 1) ** (-1/theta)

def gumbel_copula_cdf(u, v, theta):
    # C(u,v) = exp(-[(-ln u)^theta + (-ln v)^theta]^{1/theta}),  theta>=1 — верхняя хвостовая
    if theta < 1:
        theta = 1
    return np.exp(-((-np.log(u))**theta + (-np.log(v))**theta)**(1/theta))

print("Copula CDF примеры:")
print(f" Clayton(0.5,0.5, theta=2) = {clayton_copula_cdf(0.5,0.5,2):.4f}  vs independence {0.25:.4f}")
print(f" Gumbel(0.5,0.5, theta=2)  = {gumbel_copula_cdf(0.5,0.5,2):.4f}")

# Скелет MLE для theta — максимизируем правдоподобие на train парах
# Нужны маргиналы F_cioh, F_temp — оцениваем эмпирически (rank / (n+1)), затем u=F(s1), v=F(s2)
# def clayton_loglik(theta, u, v):
#     if theta <= 0: return 1e9
#     term = u**(-theta) + v**(-theta) - 1
#     logc = np.log(1+theta) + (-theta-1)*(np.log(u)+np.log(v)) + (-1/theta -2)*np.log(term)
#     return -logc.sum()
# # res = minimize(lambda th: clayton_loglik(th[0], u, v), x0=[1.0], bounds=[(0.1, 10)])
# # print("MLE theta Clayton:", res.x)

print("\nPonytail MLE — раскомментируй блок выше для подбора theta Clayton/Gumbel на train парах (§3)")
print(f"Текущий logistic-fusion PR-AUC test: {pr_auc_logreg:.4f} (NB: {pr_auc_nb:.4f}) — copula должна дать + к калибровке в хвостах")
print(f"Best operating thr (F1): {best_thr:.4f}  F1={best_f1:.4f}  P={best_prec:.4f}  R={best_rec:.4f}")
print(f"Silhouette core: {sil_core if 'sil_core' in locals() else 'n/a'}  all: {sil_all if 'sil_all' in locals() else 'n/a'}")
